In [9]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# ========== CẤU HÌNH ==========
PART_ID = 35  # <== 👈 Thay số này thành từ 1 đến 35 theo phần bạn chọn
DATA_FOLDER = "/home/trungdt2/Downloads/crawl_all_khoang_cach/part"
OUTPUT_FOLDER = os.path.join(DATA_FOLDER, f"output_part_{PART_ID:02d}")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

BATCH_SIZE = 100
SLEEP_TIME = 2
WAIT_TIME = 10

# ========== TRÌNH DUYỆT ==========
class GoogleMapsDistanceCalculator:
    def __init__(self):
        self.driver = None
        self.setup_driver()

    def setup_driver(self):
        chrome_options = webdriver.ChromeOptions()
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--headless=new")
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=chrome_options)

    def calculate_distance(self, lat1, lon1, lat2, lon2):
        try:
            url = f"https://www.google.com/maps/dir/{lat1},{lon1}/{lat2},{lon2}/data=!4m2!4m1!3e0"
            self.driver.get(url)
            time.sleep(4)
            try:
                element = WebDriverWait(self.driver, WAIT_TIME).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.xB1mrd-T3iPGc-iSfDt-ij8cu"))
                )
                return element.text
            except:
                try:
                    element = WebDriverWait(self.driver, WAIT_TIME).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "div.XdKEzd"))
                    )
                    return element.text
                except:
                    return "Không tìm thấy"
        except Exception as e:
            print(f"❌ Lỗi: {str(e)}")
            return None

    def close(self):
        if self.driver:
            self.driver.quit()

# ========== CHẠY CRAWL ==========
def crawl_with_resume(df_input):
    df = df_input.copy()
    total_rows = len(df)
    total_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

    if 'Khoảng cách đường bộ' not in df.columns:
        df['Khoảng cách đường bộ'] = None

    for batch_idx in range(total_batches):
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min((batch_idx + 1) * BATCH_SIZE, total_rows)
        output_file = os.path.join(OUTPUT_FOLDER, f"ket_qua_tu_{start_idx}_den_{end_idx - 1}.xlsx")

        if os.path.exists(output_file):
            print(f"✅ Bỏ qua batch {start_idx}-{end_idx - 1} (đã tồn tại)")
            continue

        print(f"\n🚀 Bắt đầu batch {start_idx}-{end_idx - 1}...")
        calculator = GoogleMapsDistanceCalculator()

        for idx in range(start_idx, end_idx):
            if pd.notna(df.at[idx, 'Khoảng cách đường bộ']):
                continue

            row = df.loc[idx]
            lat1, lon1 = row['VĨ ĐỘ 1'], row['KINH ĐỘ 1']
            lat2, lon2 = row['VĨ ĐỘ 2'], row['KINH ĐỘ 2']

            print(f"🔍 Dòng {idx}: ({lat1},{lon1}) → ({lat2},{lon2})")
            distance = calculator.calculate_distance(lat1, lon1, lat2, lon2)
            df.at[idx, 'Khoảng cách đường bộ'] = distance
            print(f"➡️  Kết quả: {distance}")
            time.sleep(SLEEP_TIME)

        calculator.close()
        df.iloc[start_idx:end_idx].to_excel(output_file, index=False)
        print(f"📁 Đã lưu batch vào: {output_file}")

    print("\n🎉 Đã hoàn thành tất cả các batch.")

# ========== CHẠY ==========
file_path = os.path.join(DATA_FOLDER, f"df_part_{PART_ID:02d}.pkl")
df_part = pd.read_pickle(file_path)
crawl_with_resume(df_part)


ModuleNotFoundError: No module named 'numpy._core.numeric'

In [3]:
!pip install --force-reinstall numpy

  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/ad/c9/1bf6ada582eebcbe8978f5feb26584cd2b39f94ededeea034ca8f84af8c8/numpy-2.2.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Using cached numpy-2.2.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.5
    Uninstalling numpy-2.2.5:
      Successfully uninstalled numpy-2.2.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
matplotlib 3.8.2 requires numpy<2,>=1.21, but you have numpy 2.2.5 which is incompatible.
pandas 2.1.3 requires numpy<2,>=1.23.2; python_version == "3.11", but you have numpy 2.2.5 which is incompatible.

[notice] A new release of pip is avail